<a href="https://colab.research.google.com/github/callsourav1979-personal/Assignments_HAAI-/blob/main/CV__Sorting_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!ls -lh /content/

total 4.0K
drwxr-xr-x 1 root root 4.0K Aug 18 13:44 sample_data


In [1]:
from pathlib import Path
cv_folder = Path("/content/cvs")
cv_folder.mkdir(exist_ok = True)

print(cv_folder)

/content/cvs


In [2]:
from pathlib import Path
jd_folder = Path("/content/jds")
jd_folder.mkdir(exist_ok = True)

In [ ]:
!mv /content/Resume1.pdf /content/cvs/
!mv /content/Resume2.pdf /content/cvs/
!mv /content/Resume3.docx /content/cvs/


# **Import & Test Libraries**

In [3]:
!pip install -q pypdf python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 25.3 MB/s eta 0:00:00


In [4]:
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr poppler-utils
!pip install -q pytesseract pdf2image

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package poppler-utils.
(Reading database ... 122579 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.13) ...
Setting up poppler-utils (22.02.0-2ubuntu0.13) ...
Processing triggers for man-db (2.10.2-1) ...


In [7]:
import torch
import sys
import pypdf
import docx
import pytesseract
from pdf2image import convert_from_path
from pathlib import Path

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using Device:" , device)

print("Python version:" , sys.version)
print("PyTorch version:" , torch.__version__)
print("CUDA available:" , torch.cuda.is_available())
print("PyPdf version:" , pypdf.__version__)
print("python-docx version:" , docx.__version__)
print("PyTesseract version:" , pytesseract.__version__)

if torch.cuda.is_available():
    print("CUDA version:" , torch.version.cuda)
    print("GPU device name:" , torch.cuda.get_device_name(0))
    print("GPU Memory:" , round(torch.cuda.get_device_properties(0).total_memory/1024**3,2),"GB")
else:
  print("Running on CPU")

Using Device: cuda
Python version: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
PyTorch version: 2.11.0+cu128
CUDA available: True
PyPdf version: 6.16.1
python-docx version: 1.2.0
PyTesseract version: 0.3.13
CUDA version: 12.8
GPU device name: Tesla T4
GPU Memory: 14.56 GB


In [15]:
from pypdf import PdfReader
from docx import Document
from pathlib import Path

def extract_text_from_file(file_path):
    """
    Extract text from PDF or DOCX files.

      For PDF:
          Extracts texts from all pages.

      For DOCX:
          Extracts texts from normal paragraphs and tables.

      Parameters :
           file_path(str): Path to the PDF or DOCX file.
      Returns :
           str: Extracted text from the file.
    """
    path = Path(file_path)

    if not path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    extension = path.suffix.lower()
    #---------------------------------
    # PDF
    #---------------------------------
    if extension == '.pdf':
        reader = PdfReader(str(path))

        pages = []
        for page in reader.pages:
            text = page.extract_text()
            if text:
              pages.append(text)
        return '\n'.join(pages).strip()
    #---------------------------------------
    # DOCX
    #---------------------------------------
    elif extension == '.docx':
        document = Document(str(path))

        sections = []
        # Extract normal paragraphs
        for paragraph in document.paragraphs:
            text =  paragraph.text.strip()
            if text:
              sections.append(text)

        # Extract tables
        for table in document.tables:
            sections.append("\n[TABLE ]")
            for row in table.rows:
                row_cells = []
                for cell in row.cells:
                    cell_text = cell.text.strip()
                    if cell_text:
                      row_cells.append(cell_text)
                #Combine cells in the same row
                if row_cells:
                   sections.append(" | ".join(row_cells))
            sections.append("[/TABLE]")
        #Combine paragraphs and tables
        extracted_text = "\n".join(sections).strip()

        return extracted_text

    else:
      raise ValueError(f"Unsupported file type: {extension}" "Only PDF and DOCX files are supported.")


In [16]:
from pdf2image import convert_from_path
import pytesseract
from pathlib import Path

def extract_text_from_pdf_ocr(file_path):
    """
    Extract text from scanned/image based PDF files using OCR

     Parameters :
           file_path(str): Path to the PDF file.
      Returns :
           str: OCR Extracted text from the file.
    """
    path = Path(file_path)

    if not path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    extension = path.suffix.lower()
    if extension != '.pdf':
      raise ValueError(f"This function supports PDF files only.")

    # Convert PDF pages into images
    pages = convert_from_path(str(path), dpi=300)

    extracted_pages = []

    # Process each page
    for page_number , page_image in enumerate(pages, start =1):
        print(f"Processing page {page_number}/{len(pages)}")
        #Run OCR
        text = pytesseract.image_to_string(page_image,config="--psm 6")

        #Remove unnessary whitespace
        text = text.strip()

        #Store page seperately
        page_text = (f"\n---PAGE {page_number} ---\n" f"{(text)}")

    extracted_pages.append(page_text)

    # Combine all pages
    final_text = "\n".join(extracted_pages)

    return final_text.strip()

## **`Wrapper Program`**

In [17]:
from pathlib import Path

def extract_document_text(file_path):
  """
  Main document-extraction wrapper
  Automatically selects the appropriate extraction method based on the file type
  and available text.

    Parameters:
       file_path (str) : Path to the resume file

    Returns:
       str : Extracted text from the resume
  """
  path = Path(file_path)

  #1. Check whether the file exists
  if not path.exists():
    raise FileNotFoundError(f"File not found: {file_path}")

  #2. Check supported file types
  extension = path.suffix.lower()
  if extension not in ['.pdf' , '.docx']:
    raise ValueError(f"Unsupported file type: {extension}" "Only PDF and DOCX files are supported.")

  #3 Handle PDF
  if extension == '.pdf':
    print(f"\nProcessng PDF: {path.name}")

    #First try normal PDF text extraction
    text = extract_text_from_file(path)

    #Check whether meaningful text was extracted
    if text and len(text.strip()) >= 100 :
      print("Text layer detected. Using standard PDF extraction.")
      return text.strip()
    if len(text.strip()) < 100 :
      print("Little or no text detected . Swtching to OCR ...")
      text = extract_text_from_pdf_ocr(path)
      return text.strip()

  #4 Handle DOCX
  elif extension == '.docx':
    print(f"\nProcessing DOCX: {path.name}")
    text = extract_text_from_file(path)
    return text.strip()

## **Extract Multiple CV's**

In [18]:
from pathlib import Path

def extract_multiple_cvs(cv_folder):
    """
    Extract text from all supported CV files in a folder.

    Supported formats:
      - PDF
      - DOCX

    Parameters:
       cv_folder (str) : Path to the folder containing CVs

    Returns:
       dict: Dictionary containing filename and extracted text
    """

    folder = Path(cv_folder)
    if not folder.exists():
        raise FileNotFoundError(f"CV Folder not found: {cv_folder}")

    if not folder.is_dir():
        raise ValueError(f"Path is not a directory: {cv_folder}")

    # Find all PDF and DOCX files
    cv_files = sorted(
                       [ file
                         for file in folder.iterdir()
                         if file.is_file() and file.suffix.lower() in [".pdf" , ".docx"]
                        ]
                      )
    if not cv_files:
      raise ValueError(f"No PDF or DOCX files found in :  {cv_folder}")

    cv_data = {}

    for cv_file in cv_files:
      print("="*60)
      print(f"Processing CV: {cv_file.name}")
      print("="*60)

      try:
        text = extract_document_text(cv_file)
        cv_data[cv_file.name] = text
        print(f"Characters Extracted: {len(text)}")
      except Exception as e:
        print(f"Error processing {cv_file.name}: {e}")
        cv_data[cv_file.name] = ""

    return cv_data

In [ ]:
from pathlib import Path
cv_folder = Path("/content/cvs")

for cv_file in sorted(cv_folder.iterdir()):
  if cv_file.is_file():
    text = extract_text_from_file(cv_file)
    print("\n" + "="*70)
    print("FILE:" , cv_file.name)
    print("Characters extracted :" , len(text))
    print("="*70)
    print(text[:2000])



# **Install Transformer & Model**

In [10]:
!pip install -q transformers torch accelerate


# **Model-1 Loading**

In [19]:
from transformers import AutoTokenizer,AutoModelForCausalLM
import torch

model_name1 = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer=AutoTokenizer.from_pretrained(model_name1,trust_remote_code=True)
model=AutoModelForCausalLM.from_pretrained(model_name1,
                                           torch_dtype=torch.float16 if device == "cuda" else torch.float32,
                                           trust_remote_code=True,device_map="auto")

model1=model.to(device)
print("Model loaded Successfully")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded Successfully


# **Model-2 Loading**

In [8]:
from transformers import AutoTokenizer,AutoModelForCausalLM
import torch

model_name2 = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

tokenizer2=AutoTokenizer.from_pretrained(model_name2)
model2=AutoModelForCausalLM.from_pretrained(model_name2,
                                            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
                                            device_map="auto")

#model2=model2.to(device)
print("Model loaded Successfully")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Model loaded Successfully


In [9]:
print(model2.device)
print(next(model2.parameters()).dtype)

cuda:0
torch.float16


## **READ JD from Folder**

In [20]:
jd_folder ="/content/jds"
all_jd = extract_multiple_cvs(jd_folder)
print("Total JDs processed:" , len(all_jd))

Processing CV: JD-AI Engineer.pdf

Processng PDF: JD-AI Engineer.pdf
Text layer detected. Using standard PDF extraction.
Characters Extracted: 2320
Processing CV: JD-Senior Software Engineer_v1.pdf

Processng PDF: JD-Senior Software Engineer_v1.pdf
Text layer detected. Using standard PDF extraction.
Characters Extracted: 1715
Processing CV: JD-Senior-Data-Analyst.pdf

Processng PDF: JD-Senior-Data-Analyst.pdf
Text layer detected. Using standard PDF extraction.
Characters Extracted: 4400
Processing CV: JD-SofrwareEngineer.pdf

Processng PDF: JD-SofrwareEngineer.pdf
Little or no text detected . Swtching to OCR ...
Processing page 1/1
Characters Extracted: 2530
Processing CV: JD-Technical_Architect.pdf

Processng PDF: JD-Technical_Architect.pdf
Text layer detected. Using standard PDF extraction.
Characters Extracted: 3874
Processing CV: JD-Technical_Project_Manager.pdf

Processng PDF: JD-Technical_Project_Manager.pdf
Text layer detected. Using standard PDF extraction.
Characters Extracted

In [ ]:
print(all_jd.keys())

dict_keys(['JD-AI Engineer.pdf', 'JD-Senior Software Engineer_v1.pdf', 'JD-Senior-Data-Analyst.pdf', 'JD-SofrwareEngineer.pdf', 'JD-SpecialEducator.pdf', 'JD-Technical_Project_Manager.pdf'])


In [21]:
jd_text= all_jd["JD-Technical_Architect.pdf"]
print(jd_text[:10000])

JOB DESCRIPTION
Technical Architect
Position Title:
Technical Architect
Department:
Enterprise Architecture /
Engineering
Reports To:
Director of Engineering / Chief
Architect
Employment
Type:
Full-Time
Location:
Remote / Hybrid / On-site
Experience Level:
Senior / Lead (8+ Years)
Role Overview
We are seeking a visionary and highly skilled Technical Architect to lead the design, evolution, and structural
integrity of our enterprise software ecosystems. In this role, you will bridge the gap between complex business
requirements and high-performance technical execution. You will be responsible for defining architectural
blueprints, evaluating emerging technologies, establishing robust engineering frameworks, and mentoring
development teams. The ideal candidate possesses deep distributed systems expertise, stellar communication
skills, and a proven track record of scaling cloud infrastructure to meet high enterprise demands.
Core Responsibilities
 System Blueprinting: Architect scalable,

# **JD to JSON Conversion PROMPT**

In [22]:
jd_prompt = """
DOCUMENT TYPE: JOB DESCRIPTION (JD)

You are a precise recruitment information extraction assistant.

Your task is to extract information ONLY from the provided JOB DESCRIPTION.
Do not use outside knowledge. Do not infer, assume, invent, or fabricate information.

IMPORTANT:
This is a JOB DESCRIPTION, NOT a candidate CV.

Return ONLY valid JSON.
Do not return explanations, comments, markdown, keywords, or any text outside the JSON object.

Use EXACTLY this JSON structure:

{
  "job_title": "",
  "skills": [],
  "experience": [],
  "education": [],
  "responsibilities": []
}

FIELD DEFINITIONS:

1. job_title
Extract the exact job/position title stated in the JD.

2. skills
Extract technical skills, technologies, frameworks, platforms, tools,
methodologies, architectural patterns, and certifications explicitly
mentioned as required or preferred in the JD.

Do not invent related technologies that are not explicitly mentioned.

3. experience
IMPORTANT: For a JOB DESCRIPTION, "experience" means
EMPLOYER-STATED EXPERIENCE REQUIREMENTS.

It does NOT mean candidate employment history.

Extract experience requirements such as:
- minimum total years of professional/software engineering experience
- minimum years of architectural or technical leadership experience
- required years of experience in a particular area
- required experience with particular types of systems
- required experience with methodologies or environments

Preserve the meaning and wording of the JD as closely as possible.

For example, if the JD says:
"Minimum of 8+ years of total software engineering experience"

then return:

"experience": [
  "Minimum of 8+ years of total software engineering experience"
]

If the JD says:
"at least 3+ years acting in a dedicated Architectural or Tech Lead capacity"

then return:

"experience": [
  "At least 3+ years acting in a dedicated Architectural or Tech Lead capacity"
]

NEVER convert an experience requirement into a fake employment-history object.

DO NOT create fields such as:
"title", "company", "location", "duration", or "description"
inside the JD experience array.

4. education
Extract ONLY education requirements explicitly stated in the JD.

For example, if the JD says:
"Bachelor's or Master's degree in Computer Science, Software Engineering,
or an equivalent technical field."

return the relevant education requirement using the information actually
present in the JD.

DO NOT invent:
- university names
- graduation years
- degree dates
- fields of study not stated in the JD
- candidate education details

5. responsibilities
Extract responsibilities, duties, activities, and expectations explicitly
stated in the JD.

Preserve the meaning of the JD.

CRITICAL RULES:

1. Extract information ONLY from the provided JD.
2. Do NOT use outside knowledge.
3. Do NOT infer or assume missing information.
4. Do NOT invent companies, universities, candidates, dates, job histories,
   qualifications, or other information.
5. Do NOT create candidate employment history from a JD.
6. For a JD, the "experience" field contains EMPLOYER REQUIREMENTS,
   not candidate work history.
7. For a JD, experience items must be strings, not employment-history objects.
8. For a JD, do not create "Company A", "Company B", "University X",
   or similar placeholder/fabricated values.
9. If information is not available, return [] for list fields and "" for
   the job_title field.
10. Do not output values such as "Unknown", "Not specified",
    "Not mentioned", or "None".
11. Do not include personal information that is not relevant to the
    requested extraction.
12. Do not create a "keywords" field.
13. Do not add any fields to the JSON structure.
14. Return ONLY the JSON object..

NOW EXTRACT THE INFORMATION FROM THIS JOB_DESCRIPTION:
----------------------------JOB DESCRIPTION START----------------------------------
""" + jd_text + """

----------------------------JOB DESCRIPTION END----------------------------------

"""

messages = [
      {"role": "system", "content": "You are a precise recruitment assistant that extracts information from CV"},
      {"role": "user", "content": jd_prompt}
  ]

text= tokenizer.apply_chat_template(messages , tokenize=False , add_generation_prompt= True)

inputs = tokenizer(text, return_tensors="pt").to(device)


with torch.no_grad():
  outputs = model1.generate(**inputs,max_new_tokens=500,do_sample=False)

jd_response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:],skip_special_tokens=True)

print("LLM Response:" , jd_response.strip())


LLM Response: {
  "job_title": "Technical Architect",
  "skills": [
    "Java (Spring Boot)",
    "Go",
    "Python",
    "Node.js",
    ".NET Core Enterprise stacks",
    "AWS",
    "Azure",
    "GCP",
    "Terraform/OpenTofu (IaC)",
    "Cloud Architecture Patterns",
    "Docker",
    "Kubernetes (EKS/GKE)",
    "Istio Service Mesh",
    "Helm Charts",
    "PostgreSQL",
    "MongoDB",
    "Snowflake",
    "Redis Clusters",
    "Apache Kafka",
    "RabbitMQ",
    "RESTful APIs",
    "gRPC",
    "GraphQL Topologies",
    "AWS Solutions Architect Professional",
    "Google Professional Cloud Architect",
    "Azure Solutions Architect Expert",
    "TOGAF certification"
  ],
  "experience": [
    "Minimum of 8+ years of total software engineering experience",
    "At least 3+ years acting in a dedicated Architectural or Tech Lead capacity"
  ],
  "education": [
    "Bachelor’s or Master's degree in Computer Science, Software Engineering, or an equivalent technical field"
  ],
  "responsib

# **READ CV's from Folder**

In [47]:
cv_folder ="/content/cvs"
all_cvs = extract_multiple_cvs(cv_folder)
print("Total CVs processed:" , len(all_cvs))

Processing CV: data_analyst_resume_junior.pdf

Processng PDF: data_analyst_resume_junior.pdf
Text layer detected. Using standard PDF extraction.
Characters Extracted: 1612
Processing CV: data_analyst_resume_mid_level.pdf

Processng PDF: data_analyst_resume_mid_level.pdf
Text layer detected. Using standard PDF extraction.
Characters Extracted: 1920
Processing CV: data_analyst_resume_mid_level_v2.pdf

Processng PDF: data_analyst_resume_mid_level_v2.pdf
Text layer detected. Using standard PDF extraction.
Characters Extracted: 2436
Processing CV: principal_technical_architect_cv.pdf

Processng PDF: principal_technical_architect_cv.pdf
Text layer detected. Using standard PDF extraction.
Characters Extracted: 13168
Processing CV: sales_director_cv.pdf

Processng PDF: sales_director_cv.pdf
Text layer detected. Using standard PDF extraction.
Characters Extracted: 2726
Processing CV: software_engineer_resume_1.pdf

Processng PDF: software_engineer_resume_1.pdf
Text layer detected. Using standar

In [ ]:
print(all_cvs.keys())

In [48]:
cv_text= all_cvs["sales_director_cv.pdf"]
print(cv_text[:10000])

MARCUS VANCE
 SALES DIRECTOR  Global Enterprise Commercial Strategy
 New York, NY  m.vance@email.com  +1 (555) 019-2834  linkedin.com/in/marcusvance-sales
EXECUTIVE PROFILE
High-impact Sales Director with over 15 years of success driving multi-million dollar revenue growth across global commercial
markets. Recognized for constructing elite corporate sales forces, engineering high-yield territory models, and cementing
top-tier enterprise relationships. Expert in commercial revenue maximization, market penetration strategies, and navigating
complex corporate procurement ecosystems to consistently outperform organizational growth targets.
CORE COMMERCIAL COMPETENCIES
 Enterprise Account Acquisition
 Revenue Pipeline Optimization
 Key Stakeholder Relations
 High-Value Contract Negotiation
 Sales Force Commission Structures
 Market Share Expansion
 Cross-Functional Team Leadership
 Executive Client Advisory
 P&L & Budget Accountability
PROFESSIONAL EXPERIENCE
Sales Director | A

# **CV to JSON Conversion PROMPT**

In [49]:
cv_prompt = """
You are an expert recruitment assistant.

Your task is to analyze the following CV/Resume and convert the information explicitly stated in it into a structured JSON format.

JOB DESCRIPTION :
""" + cv_text + """

Extract the following information:
1. Candidate Name
2. Job title or professional title, if explicitly stated
3. Skills explicitly mentioned in the CV
4. Work experience explicitly mentioned in the CV
5. Education/qualifications explicitly mentioned in the CV
6. Job responsibilities , duties , projects , or work activities explicitly mentioned in the CV


Return ONLY vaid JSON in exactly this format:

{
  "candidate_name": "",
  "job_title": "",
  "skills": [],
  "experience": [],
  "education": [],
  "responsibilities": []
}

IMPORTANT RULES:

1. Extract information only from the CV. Do not use outside
   knowledge or assumptions.

2. Preserve the original meaning and wording of the Job Description
   as closely as possible.

3. Do NOT classify anything as required, preferred, or other at this stage.

4. "skills" must contain actual skills, abilities, knowledge, tools,
   technologies, software, programming languages, or competencies
   explicitly mentioned in the Job Description.

5. "experience" must contain explicitly stated experience requirements
   or experience statements.

6. "education" must contain explicitly stated educational qualifications,
   degrees, diplomas, certifications, or fields of study.

7. "responsibilities" must contain the actual duties and responsibilities
   stated in the Job Description.

8. Do NOT convert responsibilities into skills.

9. Do NOT convert education into skills.

10. Do NOT convert experience into skills.

11. Keep responsibilities as complete statements. Do not unnecessarily
    summarize or omit important responsibilities.

12. If information is not present, return an empty list [].

13. NEVER output "None specified", "Not specified", "Not mentioned",
    "Unknown", or similar text. Use [] instead.

14. Return ONLY the JSON object in exactly the structured mentioned . Do not provide explanations,
    comments, markdown, or text outside the JSON.

15. Do NOT add any fields that are not present in the JSON structure provided. Do NOT generate keywords , summary , profile or
    any additional fields.

16. Stop generating immediately after the closing ) of the JSON object.
"""

messages = [
      {"role": "system", "content": "You are a precise recruitment assistant that extracts information from CV"},
      {"role": "user", "content": cv_prompt}
  ]

text= tokenizer.apply_chat_template(messages , tokenize=False , add_generation_prompt= True)

inputs = tokenizer(text, return_tensors="pt").to(device)


with torch.no_grad():
  outputs = model1.generate(**inputs,max_new_tokens=1000,do_sample=False)

cv_response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:],skip_special_tokens=True)

print("LLM Response:" , cv_response.strip())


LLM Response: ```json
{
  "candidate_name": "MARCUS VANCE",
  "job_title": "SALES DIRECTOR",
  "skills": [
    "Enterprise Account Acquisition",
    "Revenue Pipeline Optimization",
    "Key Stakeholder Relations",
    "High-Value Contract Negotiation",
    "Sales Force Commission Structures",
    "Market Share Expansion",
    "Cross-Functional Team Leadership",
    "Executive Client Advisory",
    "P&L & Budget Accountability"
  ],
  "experience": [
    {
      "company": "Apex Global Solutions",
      "position": "Sales Director",
      "duration": "2022 – Present",
      "responsibilities": [
        "Direct all commercial activity and pipeline development for a $45M enterprise business segment.",
        "Formulated an aggressive market-entry strategy that grew market share by 24% within 18 months.",
        "Championed new incentive models for a team of 35 account executives, elevating average quota attainment from 72% to 91%.",
        "Spearheaded contract negotiations for 12 Fo

# **Clean JSON Output**

In [44]:
import json

def clean_and_validate_json(response):
  response = response.strip()
  # Remove opening markdown fence
  if response.startswith("```json"):
    response = response[len("```json"):].strip()
  elif response.startswith("```"):
    response = response[len("```"):].strip()

  # Remove closing markdown fence
  if response.endswith("```"):
    response = response[:-3].strip()

  # Validate JSON
  return json.loads(response)

In [50]:
jd_json = clean_and_validate_json(jd_response)
cv_json = clean_and_validate_json(cv_response)
jd_text =json.dumps(jd_json,indent=2)
cv_text =json.dumps(cv_json,indent=2)
print(jd_text)
print("*"*60)
print(cv_text)

{
  "job_title": "Technical Architect",
  "skills": [
    "Java (Spring Boot)",
    "Go",
    "Python",
    "Node.js",
    ".NET Core Enterprise stacks",
    "AWS",
    "Azure",
    "GCP",
    "Terraform/OpenTofu (IaC)",
    "Cloud Architecture Patterns",
    "Docker",
    "Kubernetes (EKS/GKE)",
    "Istio Service Mesh",
    "Helm Charts",
    "PostgreSQL",
    "MongoDB",
    "Snowflake",
    "Redis Clusters",
    "Apache Kafka",
    "RabbitMQ",
    "RESTful APIs",
    "gRPC",
    "GraphQL Topologies",
    "AWS Solutions Architect Professional",
    "Google Professional Cloud Architect",
    "Azure Solutions Architect Expert",
    "TOGAF certification"
  ],
  "experience": [
    "Minimum of 8+ years of total software engineering experience",
    "At least 3+ years acting in a dedicated Architectural or Tech Lead capacity"
  ],
  "education": [
    "Bachelor\u2019s or Master's degree in Computer Science, Software Engineering, or an equivalent technical field"
  ],
  "responsibilities":

# Matching **PROMPT**

In [51]:
matching_prompt = f"""
You are an expert technical recruitment evaluator.

Your task is to compare a JOB DESCRIPTION (JD) with a CANDIDATE CV and determine how well the candidate matches the job.

Evaluate ONLY the information provided in the JD and CV JSON.
Do NOT invent, assume, or add skills, experience, qualifications, or technologies that are not supported by the provided data.

IMPORTANT SCORING RULES:

1. skill_match_score:
   Evaluate how well the candidate's skills match the required skills in the JD.

2. experience_match_score:
   Evaluate how well the candidate's professional experience and responsibilities match the experience and responsibilities required by the JD.

3. education_match_score:
   Evaluate how well the candidate's education matches the education requirements in the JD.
   If the JD does not specify an education requirement, do not automatically give 100. Use the available information and apply reasonable judgment based only on the provided data.

4. overall_score:
   Calculate an overall assessment based on skills, experience, education, and responsibilities.
   The overall score must reflect the actual evidence in the JD and CV.
   Do not automatically use the same score for all categories.

5. All scores must be integers between 0 and 100.

6. Do not give a high score simply because the candidate has some overlapping skills.
   Consider the importance and relevance of the skills and experience to the actual JD.

7. A candidate with substantial missing mandatory or core skills should receive a significantly lower score.

8. A candidate with strong alignment across the core skills and responsibilities should receive a higher score.

MATCHED SKILLS RULES:

9. matched_skills must contain ONLY skills that are supported by both the JD and the candidate CV.

10. matched_skills must contain UNIQUE items only.

11. NEVER repeat the same skill.

12. Maximum 15 matched skills.

13. Prefer the most relevant and important matching skills rather than listing every minor overlap.

MISSING SKILLS RULES:

14. missing_skills must contain ONLY important skills or technologies required by the JD that are not supported by the candidate CV.

15. Do NOT include a skill in missing_skills if it is clearly present in the candidate CV.

16. missing_skills must contain UNIQUE items only.

17. NEVER repeat the same skill.

18. Maximum 15 missing skills.

19. If there are no important missing skills, return an empty array.

EXPERIENCE RULES:

20. relevant_experience must contain concise statements showing how the candidate's experience matches the JD.

21. Use ONLY evidence available in the candidate CV.

22. Do not invent responsibilities or achievements.

23. Maximum 5 relevant experience items.

24. Each item must be UNIQUE and concise.

RECOMMENDATION RULES:

25. Use one of the following recommendations:

"Strong Match"
"Good Match"
"Partial Match"
"Poor Match"

26. Strong Match means the candidate has strong alignment with the core skills and experience required by the JD.

27. Good Match means the candidate has substantial alignment but has some gaps.

28. Partial Match means the candidate has some relevant skills or experience but significant gaps exist.

29. Poor Match means the candidate has little relevant alignment with the JD.

OUTPUT RULES:

30. Return ONLY valid JSON.

31. Do NOT return markdown.

32. Do NOT use ```json or ```.

33. Do NOT provide any explanation before or after the JSON.

34. Do NOT repeat any JSON field.

35. Do NOT repeat any array item.

36. Stop immediately after the closing bracket of the JSON object.

RETURN EXACTLY THIS STRUCTURE:

{{
  "candidate_name": "",
  "overall_score": 0,
  "skill_match_score": 0,
  "experience_match_score": 0,
  "education_match_score": 0,
  "matched_skills": [],
  "missing_skills": [],
  "relevant_experience": [],
  "recommendation": ""
}}

JOB DESCRIPTION JSON:

"""+ jd_text +"""

CANDIDATE CV JSON:

"""+ cv_text +"""


"""

messages2 = [
      {"role": "system", "content": "You are a precise recruitment matching assistant"},
      {"role": "user", "content": matching_prompt}
  ]

text2= tokenizer2.apply_chat_template(messages2 , tokenize=False , add_generation_prompt= True)

inputs2 = tokenizer2(text2, return_tensors="pt").to(device)


with torch.no_grad():
  outputs2 = model2.generate(**inputs2,max_new_tokens=1000,do_sample=False)

match_response = tokenizer2.decode(outputs2[0][inputs2["input_ids"].shape[1]:],skip_special_tokens=True)

print("LLM #2 Response:" , match_response.strip())


LLM #2 Response: {
  "candidate_name": "MARCUS VANCE",
  "overall_score": 85,
  "skill_match_score": 90,
  "experience_match_score": 80,
  "education_match_score": 85,
  "matched_skills": [
    "Enterprise Account Acquisition",
    "Revenue Pipeline Optimization",
    "Cross-Functional Team Leadership",
    "High-Value Contract Negotiation",
    "Sales Force Commission Structures",
    "Market Share Expansion",
    "Cross-Functional Team Leadership",
    "Executive Client Advisory",
    "P&L & Budget Accountability"
  ],
  "missing_skills": [
    "Cloud Architecture Patterns",
    "Terraform/OpenTofu (IaC)",
    "AWS Solutions Architect Professional",
    "Google Professional Cloud Architect",
    "Azure Solutions Architect Expert",
    "TOGAF certification"
  ],
  "relevant_experience": [
    "System Blueprinting: Architect scalable, highly available, fault-tolerant, and secure distributed software and cloud infrastructures.",
    "Technology Governance: Define and enforce technical r